# Lab 6 - SQL with DuckDB

<a href="https://colab.research.google.com/github/gaulinmp/AccountingDataAnalytics/blob/main/labs_hw/week6_ETL-SQL-LLM/Lab-6_SQL-with-DuckDB.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This notebook will walk you through loading two datasets and joining them together with SQL. We will cover the following steps:

1. Loading the Compustat (annual financial statements) and CRSP (daily stock prices) data
2. Querying the data with SQL, using a library called DuckDB

The intent of the notebook is for you to execute each cell in order. Before you start, drag both `lab6_data_compustat.csv` and `lab6_data_crsp.csv` into your Colab page (see Lab 1 for instructions).

In [ ]:
# The usual imports: pathlib to point to files, and pandas to load them
from pathlib import Path
import pandas as pd
# This one is new. DuckDB is a small, fast database that lives inside Python.
# It lets us write SQL queries directly against pandas DataFrames.
import duckdb

In [ ]:
# Point to the files, and make sure they exist
compustat_path = Path("lab6_data_compustat.csv")
crsp_path = Path("lab6_data_crsp.csv")
compustat_path.exists(), crsp_path.exists()

That should say `(True, True)` above. If either says `False`, then that file is not loaded correctly. Please make sure to drag both CSV files into your Colab page.

Okay, now to actually load the data. Notice the `parse_dates` argument: it tells pandas which columns are dates, so they get loaded as dates instead of text. That matters later, because we're going to compare dates in our SQL query.

In [ ]:
# Load Compustat: one row per company per fiscal year
compustat = pd.read_csv(compustat_path, parse_dates=["fiscal_year_end", "fiscal_year_end_prev", "fiscal_year_end_next",
                                                     "earn_annc_date", "earn_annc_date_prev", "earn_annc_date_next"])
compustat.head()

In [ ]:
# Load CRSP: one row per company per trading day
crsp = pd.read_csv(crsp_path, parse_dates=["date"])
crsp.head()

Take a second to notice how different these two tables are. Compustat has one row per firm-*year* (a few hundred rows), while CRSP has one row per firm-*day* (over 150,000 rows). Let's confirm:

In [ ]:
print("Compustat:", compustat.shape)
print("CRSP:     ", crsp.shape)

Both tables have a `permno` column, which is CRSP's permanent identifier for a stock. That's our key for linking the two tables together. (Why not `ticker`? Because tickers change. American Airlines was `AMR` before it was `AAL`, but its `permno` stayed the same.)

## SQL with DuckDB

Now for the fun part. We want to match each firm's *daily* stock data to the *annual* financial statements that were current on that day. In other words, for each fiscal year, grab every trading day from that fiscal year end up to (but not including) the next one.

That's a join on `permno`, plus a date range condition. Here's the SQL:

```sql
-- Example Query: Join Compustat and CRSP on Permno and Date between fiscal year end dates
SELECT *
FROM compustat
JOIN crsp
    ON compustat.permno = crsp.permno
        AND date >= fiscal_year_end
        AND date < fiscal_year_end_next
LIMIT 5
```

The neat trick with DuckDB is that the table names in the query (`compustat` and `crsp`) are just the names of our pandas DataFrames. DuckDB finds them automatically. We write the query as a Python string, hand it to `duckdb.sql()`, and convert the result back to a DataFrame with `.df()`.

In [ ]:
query = """
-- Example Query: Join Compustat and CRSP on Permno and Date between fiscal year end dates
SELECT *
FROM compustat
JOIN crsp
    ON compustat.permno = crsp.permno
        AND date >= fiscal_year_end
        AND date < fiscal_year_end_next
LIMIT 5
"""

duckdb.sql(query).df()

Scroll all the way to the right in that output. You'll see columns named `permno_1` and `ticker_1`. That's because both tables have a `permno` and `ticker` column, and `SELECT *` grabs everything from both. DuckDB renames the duplicates so they don't collide.

In practice, you'd usually pick just the columns you want instead of `*`. Here's the same join, returning only the firm name, fiscal year, net income (`ni`), the trading date, and the daily return (`ret`). Notice the `table.column` syntax, which tells SQL which table each column comes from.

In [ ]:
query = """
SELECT compustat.name, compustat.fyear, compustat.ni, crsp.date, crsp.ret
FROM compustat
JOIN crsp
    ON compustat.permno = crsp.permno
        AND date >= fiscal_year_end
        AND date < fiscal_year_end_next
LIMIT 5
"""

duckdb.sql(query).df()

Now you're writing SQL. From here, you can add a `WHERE` clause to filter, `GROUP BY` to aggregate (e.g., the average daily return per firm-year), or `ORDER BY` to sort. And remember: if you get stuck, write a comment describing what you want, and let Gemini suggest the query.